# Model calibration

*añadir explicación*


### Set up

In [28]:
# load the required libraries
using DataFrames
using CSV
using Statistics
using Distributions
using StatsBase
using JuMP
using Gurobi

project_root = dirname(@__DIR__)

# load the function that models the electricity market and the other auxiliary functions
include(joinpath(project_root, "scripts", "model_electricity_market.jl"))
include(joinpath(project_root, "scripts", "auxiliary_functions.jl"))

# load the fixed datasets
historical_data        = CSV.read(joinpath(project_root, "data", "historical_data.csv"), DataFrame, delim=',');
technology_data        = CSV.read(joinpath(project_root, "data", "technology_data.csv"), DataFrame, delim=',');
projection_deltas_data = CSV.read(joinpath(project_root, "data", "projection_deltas_data.csv"), DataFrame, delim=',');

In [29]:
# reescalo valores mientras no hemos actualizado el csv original
for col in names(historical_data)
    col_str = string(col)
    if col_str == "spot_price_eur_gwh"
        # historical_data[!, col] = historical_data[!, col] ./ 1000
        continue
    elseif endswith(col_str, "_eur_gwh") 
        historical_data[!, col] = historical_data[!, col] .* 1000
    elseif endswith(col_str, "_gwh") || endswith(col_str, "_gw")
        historical_data[!, col] = historical_data[!, col] ./ 1000
    end
end

first(historical_data, 10)

Row,time_long,year,month,day,hour,spot_price_eur_gwh,residential_demand_gwh,commercial_demand_gwh,industrial_demand_gwh,coal_cap_gw,combined_cycle_cap_gw,gas_turbine_cap_gw,vapor_turbine_cap_gw,cogeneration_cap_gw,diesel_cap_gw,nonrenewable_waste_cap_gw,nuclear_cap_gw,conventional_hydro_cap_gw,run_of_river_hydro_cap_gw,pumped_hydro_turb_cap_gw,pumped_hydro_pump_cap_gw,solar_pv_cap_gw,solar_thermal_cap_gw,wind_cap_gw,other_renewable_cap_gw,renewable_waste_cap_gw,batteries_cap_gw,coal_gen_gwh,combined_cycle_gen_gwh,gas_turbine_gen_gwh,vapor_turbine_gen_gwh,cogeneration_gen_gwh,diesel_gen_gwh,nonrenewable_waste_gen_gwh,nuclear_gen_gwh,conventional_hydro_gen_gwh,run_of_river_hydro_gen_gwh,pumped_hydro_gen_gwh,pumped_hydro_consumption_gwh,solar_pv_gen_gwh,solar_thermal_gen_gwh,wind_gen_gwh,other_renewable_gen_gwh,renewable_waste_gen_gwh,auxiliary_generation_gen_gwh,batteries_gen_gwh,nuclear_cap_factor,hydro_cap_factor,solar_pv_cap_factor,solar_thermal_cap_factor,wind_cap_factor,imports_France_gwh,exports_France_gwh,net_flows_France_gwh,imports_Portugal_gwh,exports_Portugal_gwh,net_flows_Portugal_gwh,imports_Morocco_gwh,exports_Morocco_gwh,net_flows_Morocco_gwh,cost_coal_eur_gwh,cost_gas_eur_gwh,cost_diesel_eur_gwh,cost_uranium_eur_gwh,eu_ets_price_eur_tco2,eff_65_Average,eff_70_Average,eff_75_Average,eff_80_Average,eff_85_Average
,String31,Int64,Int64,Int64,Int64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Int64,Float64,Float64,Float64,Float64,Float64,Float64,Float64
1,2020-01-01 00:00:00 UTC,2020,1,1,0,38.6,14.2244,5.01241,4.39883,9.45625,26.2501,1.14865,0.48264,5.70184,0.76867,0.441993,7.11729,13.655,3.453,4.068,2.8476,8.94187,2.30222,25.7213,1.0964,0.157322,0.025,0.435,5.45133,0.01025,0.144833,2.45087,0.226083,0.165639,7.097,4.829,1.425,0.0,0.471,0.0168333,0.000833333,1.66792,0.3855,0.0897475,0.0,0.0,0.999987,0.813006,0.00188253,0.000361969,0.0648456,0.0,0.34461,-0.34461,0.0,0.28542,-0.28542,0.35839,0.0,0.35839,7223.82,11280.0,66000,1692.0,24.06,145.448,139.568,134.2,128.992,124.352
2,2020-01-01 01:00:00 UTC,2020,1,1,1,36.55,12.9226,4.96238,4.42463,9.45625,26.2501,1.14865,0.48264,5.70184,0.76867,0.441993,7.11729,13.655,3.453,4.068,2.8476,8.94187,2.30222,25.7213,1.0964,0.157322,0.025,0.420333,5.07717,0.0123333,0.14375,2.44567,0.195167,0.167191,7.09733,4.558,1.419,0.0,0.516,0.0168333,0.000916667,1.72925,0.378583,0.09001,0.0,0.0,0.999987,0.813006,0.00188253,0.000398166,0.0672302,0.0,0.34311,-0.34311,0.0,0.79464,-0.79464,0.22069,0.0,0.22069,7223.82,11280.0,66000,1692.0,24.06,45.576,42.104,39.432,37.024,34.616
3,2020-01-01 02:00:00 UTC,2020,1,1,2,32.32,11.6723,4.92436,4.43118,9.45625,26.2501,1.14865,0.48264,5.70184,0.76867,0.441993,7.11729,13.655,3.453,4.068,2.8476,8.94187,2.30222,25.7213,1.0964,0.157322,0.025,0.392333,4.76808,0.0119167,0.127333,2.43002,0.17975,0.171859,7.09792,4.377,1.402,0.0,1.038,0.0166667,0.000833333,1.72983,0.377083,0.089524,0.0,0.0,0.999987,0.813006,0.00186389,0.000361969,0.0672529,0.0,0.62778,-0.62778,0.0,0.75436,-0.75436,0.26925,0.0,0.26925,7223.82,11280.0,66000,1692.0,24.06,12.624,11.192,10.28,9.376,8.856
4,2020-01-01 03:00:00 UTC,2020,1,1,3,30.85,10.8474,4.91447,4.44178,9.45625,26.2501,1.14865,0.48264,5.70184,0.76867,0.441993,7.11729,13.655,3.453,4.068,2.8476,8.94187,2.30222,25.7213,1.0964,0.157322,0.025,0.38825,4.96175,0.012,0.120083,2.42439,0.16875,0.170384,7.09758,4.482,1.396,0.0,1.968,0.017,0.00141667,1.69192,0.369667,0.0902785,0.0,0.0,0.999987,0.813006,0.00190117,0.000615347,0.0657787,0.0,1.0962,-1.0962,0.0,0.32635,-0.32635,0.23025,0.0,0.23025,7223.82,11280.0,66000,1692.0,24.06,6.968,6.288,5.616,4.

### Running the model for historical data

Instead of projecting the historical dataset to the future as in the MC loop, we just run it once to 

In [30]:
# set parameters
baseline_years = [2023, 2024] 

scenarios = DataFrame(
    scenario_name         = ["baseline",       "nuclear",      "optimistic",  "climate change"],
    
    elas_anomaly          = [1.0,             1.0,            2.0,            1.0],
    hydro_anomaly         = [1.0,             1.0,            1.0,            0.8],

    coal_phase_out        = [true,            true,           true,          true],
    nuclear_phase_out     = [true,            false,          true,          true],
    batt_cap_multiplier   = [1.0,             0.75,           1.25,           1.0],
    ren_cap_multiplier    = [1.0,             0.9,            1.1,            1.0]
)

scenario_names = scenarios.scenario_name

scenario_dict = Dict(
    scen => NamedTuple(scenarios[i, :])
    for (i, scen) in enumerate(scenario_names)
)

scenario_params = scenario_dict["baseline"]

technical_params = (
    elas_residential = 0.015,
    elas_commercial = 0.03,
    elas_industrial = 0.05,
    grid_loss_factor = 0.015,
    coal_min = 0.15,
    coal_max = 0.65,
    coal_ramp = 0.05,
    ccgt_min = 0.05,
    ccgt_max = 1.0,
    ccgt_ramp = 0.25,
    cogen_min = 0.15,
    cogen_max = 0.6,
    cogen_ramp = 0.1,
    diesel_min = 0.2,
    non_ren_waste_min = 0.0,
    non_ren_waste_max = 0.6,
    non_ren_waste_ramp = 0.01,
    conv_hydro_ramp = 0.1,
    ror_lo_high = 0.3,
    ror_hi_high = 0.5,
    ror_lo_med_high = 0.2,
    ror_hi_med_high = 0.4,
    ror_lo_med_low = 0.15,
    ror_hi_med_low = 0.3,
    ror_lo_low = 0.1,
    ror_hi_low = 0.15,
    ror_ramp = 0.2,
    ph_storage_cap_gwh = 50.0,
    ph_roundtrip_eff = 0.75,
    batt_roundtrip_eff = 0.9,
    # batt_self_discharge = 0.002 / 24,
    batt_duration = 4.0,
    other_ren_min = 0.25,
    other_ren_max = 0.6,
    other_ren_ramp = 0.05,
    ren_waste_min = 0.0,
    ren_waste_max = 0.65,
    ren_waste_ramp = 0.05
    )

    
# sample time window from historical data (could skip and do full year)
sampled_window_data = sample_time_window(historical_data, baseline_years)

# compute iteration-specific parameters
iteration_params = compute_iteration_params(
        projected  = sampled_window_data,    # sampled window of historical (not projected) data
        technology = technology_data,        # fixed technical and economic parameters by generation technology
        technical  = technical_params,       # model calibration parameters shared across scenarios
        scenario   = scenario_params         # set to the baseline case
        )

# solve the model
results = dispatch_electricity_market(
        projected  = sampled_window_data,    # hourly projected data for 2030
        technology = technology_data,        # fixed technical and economic parameters by generation technology
        technical  = technical_params,       # model calibration parameters shared across scenarios
        scenario   = scenario_params,        # scenario-specific parameters
        iteration  = iteration_params        # iteration-specific parameters
        )

Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2809983
Academic license 2809983 - for non-commercial use only - registered to pa___@bse.eu
Set parameter OutputFlag to value 1
Set parameter TimeLimit to value 300
Set parameter MIPGap to value 0.03
Set parameter MIPGap to value 0.03
Set parameter TimeLimit to value 300
Set parameter OutputFlag to value 1
Gurobi Optimizer version 12.0.0 build v12.0.0rc1 (mac64[x86] - Darwin 21.6.0 21H1320)

CPU model: Intel(R) Core(TM) i5-5250U CPU @ 1.60GHz
Thread count: 2 physical cores, 4 logical processors, using up to 4 threads

Non-default parameters:
TimeLimit  300
MIPGap  0.03

Academic license 2809983 - for non-commercial use only - registered to pa___@bse.eu
Optimize a model with 133052 rows, 86688 columns and 332604 nonzeros
Model fingerprint: 0x225f2a37
Model has 4032 quadratic constraints
Coefficient statistics:
  Matrix range     [1e-03, 3e+05]
  QMatrix range    [7e-01, 9e+02]
  QLMatrix range   [1e+00, 1

Dict{String, Any} with 55 entries:
  "lifecycle_emissions"    => [1264.55, 1264.55, 1264.55, 1264.55, 1264.55, 126…
  "total_demand"           => [24.32, 22.7848, 21.8484, 21.402, 21.3361, 21.716…
  "non_renewable_gen"      => [9.73438, 9.73438, 9.73438, 9.73438, 9.73438, 9.7…
  "pumped_hydro_pumping"   => [0.434364, 0.649549, 0.650509, 0.650669, 0.651127…
  "battery_storage"        => [0.0395152, 0.042996, 0.0465525, 0.0505695, 0.054…
  "solar_pv_gen"           => [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.1055, 3.104…
  "exports_POR"            => [0.69255, 0.551175, 0.487355, 0.698975, 0.876415,…
  "commercial_demand"      => [6.34891, 6.19503, 6.1028, 6.03847, 5.994, 6.0824…
  "max_price"              => 3313.18
  "diesel_gen"             => [0.153734, 0.153734, 0.153734, 0.153734, 0.153734…
  "renewable_waste_gen"    => [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0…
  "exports_MOR"            => [0.0, 0.0, 0.0, 0.0, 0.018785, 0.0087425, 0.01185…
  "curtailment_wind"       => 0.0099

### Compare model results with historical data

Esto tendría que revisarlo todo

In [9]:
function compute_stats(data, vars; prefix="")
    out = DataFrame(variable = String[])
    
    for var in vars
        values = data isa Dict ? data[var] : data[!, var]
        
        push!(out, (
            variable = var,
            min = minimum(values),
            mean = mean(values),
            median = median(values),
            max = maximum(values),
            std = std(values)
        ))
    end
    
    rename!(out, names(out) .=> (n -> n == "variable" ? n : n * "_" * prefix))
    return out
end

# variables por las que queremos comprobar resultados
variables = [
    "price", "total_demand", "residential_demand", "commercial_demand", "industrial_demand",
    "total_generation",
    "coal_gen", "combined_cycle_gen", "cogeneration_gen", "diesel_gen", "non_renewable_waste_gen",
    "nuclear_gen", "conventional_hydro_gen", "run_of_river_hydro_gen",
    "pumped_hydro_out", "solar_pv_gen", "solar_thermal_gen", "wind_gen",
    "other_renewable_gen", "renewable_waste_gen",
    "battery_out",
    "imports_FRA", "imports_POR", "imports_MOR",
    "share_renewable_gen"
]


model_stats = compute_stats(results, variables; prefix="model")

# Generación total
generation_cols = [
    :coal_gen_gwh, :combined_cycle_gen_gwh, :gas_turbine_gen_gwh, :vapor_turbine_gen_gwh,
    :cogeneration_gen_gwh, :diesel_gen_gwh, :non_renewable_waste_gen_gwh,
    :nuclear_gen_gwh, :conventional_hydro_gen_gwh, :run_of_river_hydro_gen_gwh,
    :pumped_hydro_out_gwh,
    :solar_pv_gen_gwh, :solar_thermal_gen_gwh, :wind_gen_gwh,
    :other_renewable_gen_gwh, :renewable_waste_gen_gwh,
    :battery_out_gwh
]

sampled_window_data.total_generation_gwh =
    [sum(row) for row in eachrow(select(sampled_window_data, generation_cols))]

# Share renovables
renewable_cols = [
    :conventional_hydro_gen_gwh, :run_of_river_hydro_gen_gwh,
    :pumped_hydro_out_gwh,
    :solar_pv_gen_gwh, :solar_thermal_gen_gwh, :wind_gen_gwh,
    :other_renewable_gen_gwh, :renewable_waste_gen_gwh
]

sampled_window_data.share_renewable_gen =
    [total > 0 ? sum(row[renewable_cols]) / total : 0
     for (row, total) in zip(eachrow(sampled_window_data), sampled_window_data.total_generation_gwh)]

var_map = Dict(
    "price" => :spot_price_eur_mwh,  # aquí ojo: sigue en €/MWh
    "total_demand" => :total_demand_gwh,
    "residential_demand" => :residential_demand_gwh,
    "commercial_demand" => :commercial_demand_gwh,
    "industrial_demand" => :industrial_demand_gwh,
    "total_generation" => :total_generation_gwh,
    "coal_gen" => :coal_gen_gwh,
    "combined_cycle_gen" => :combined_cycle_gen_gwh,
    "cogeneration_gen" => :cogeneration_gen_gwh,
    "diesel_gen" => :diesel_gen_gwh,
    "non_renewable_waste_gen" => :non_renewable_waste_gen_gwh,
    "nuclear_gen" => :nuclear_gen_gwh,
    "conventional_hydro_gen" => :conventional_hydro_gen_gwh,
    "run_of_river_hydro_gen" => :run_of_river_hydro_gen_gwh,
    "pumped_hydro_out" => :pumped_hydro_out_gwh,
    "solar_pv_gen" => :solar_pv_gen_gwh,
    "solar_thermal_gen" => :solar_thermal_gen_gwh,
    "wind_gen" => :wind_gen_gwh,
    "other_renewable_gen" => :other_renewable_gen_gwh,
    "renewable_waste_gen" => :renewable_waste_gen_gwh,
    "battery_out" => :battery_out_gwh,
    "imports_FRA" => :imports_France_gwh,
    "imports_POR" => :imports_Portugal_gwh,
    "imports_MOR" => :imports_Morocco_gwh,
    "share_renewable_gen" => :share_renewable_gen
)

real_stats = compute_stats(
    sampled_window_data,
    [var_map[v] for v in variables];
    prefix="real"
)

real_stats.variable = variables

combined_stats = innerjoin(model_stats, real_stats, on="variable")

ArgumentError: ArgumentError: row insertion with `cols` equal to `:setequal` requires `row` to have the same number of elements as the number of columns in `df`.

### Gráficos de comparativa

A partir de aquí también tendría que revisarlo todo

In [23]:
# Filter the results dictionary to include only the specified keys, to being able to make the graphs
filtered_results = Dict(
    "residential_demand" => results["residential_demand"],
    "commercial_demand" => results["commercial_demand"],
    "industrial_demand" => results["industrial_demand"],
    "price" => results["price"],
    "coal_gen" => results["coal_gen"],
    "combined_cycle_gen" => results["combined_cycle_gen"],
    "gas_turbine_gen" => results["gas_turbine_gen"],
    "vapor_turbine_gen" => results["vapor_turbine_gen"],
    "cogeneration_gen" => results["cogeneration_gen"],
    "diesel_gen" => results["diesel_gen"],
    "non_renewable_waste_gen" => results["non_renewable_waste_gen"],
    "conventional_hydro_gen" => results["conventional_hydro_gen"],
    "run_of_river_hydro_gen" => results["run_of_river_hydro_gen"],
    "pumped_hydro_gen" => results["pumped_hydro_gen"],
    "nuclear_gen" => results["nuclear_gen"],
    "solar_pv_gen" => results["solar_pv_gen"],
    "solar_thermal_gen" => results["solar_thermal_gen"],
    "wind_gen" => results["wind_gen"],
    "other_renewable_gen" => results["other_renewable_gen"],
    "renewable_waste_gen" => results["renewable_waste_gen"],
    "battery_gen" => results["battery_gen"],
    "imports_FRA" => results["imports_FRA"],
    "imports_POR" => results["imports_POR"],
    "imports_MOR" => results["imports_MOR"],
    "exports_FRA" => results["exports_FRA"],
    "exports_POR" => results["exports_POR"],
    "exports_MOR" => results["exports_MOR"],
    "share_renewable_gen" => results["share_renewable_gen"],
    )

# Convert the filtered results dictionary into a DataFrame
results_df = DataFrame(filtered_results)

# Add a time column for better visualization
results_df[!, :time] = 1:size(results_df, 1)




UndefVarError: UndefVarError: `results` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [24]:
# Plot actual price vs model-solved price across time
plot(1:length(toy_data.spot_price_eur_mwh), toy_data.spot_price_eur_mwh, 
  label="Actual Price (€/MWh)", xlabel="Time (hours)", ylabel="Price (€/MWh)", legend=:topright)
plot!(results_df[!, :time], results_df[!, :price], label="Model-Solved Price (€/MWh)")

UndefVarError: UndefVarError: `toy_data` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [25]:
# Determine the common y-axis range
y_min = min(minimum(toy_data.residential_demand_mwh), minimum(toy_data.industrial_demand_mwh))
y_max = max(maximum(toy_data.residential_demand_mwh), maximum(toy_data.industrial_demand_mwh))

# Plot actual vs model-solved residential demand
plot(1:length(toy_data.residential_demand_mwh), toy_data.residential_demand_mwh, 
    label="Actual residential demand (MWh)", xlabel="Time (hours)", ylabel="Demand (MWh)", legend=:topright, 
    title="Residential Demand", ylim=(y_min, y_max), yformatter = :plain)
plot!(results_df[!, :time], results_df[!, :residential_demand], label="Model-Solved residential demand (MWh)")

UndefVarError: UndefVarError: `toy_data` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [26]:
#Plot actual vs model-solved commercial demand
plot(1:length(toy_data.commercial_demand_mwh), toy_data.commercial_demand_mwh, 
    label="Actual commercial demand (MWh)", xlabel="Time (hours)", ylabel="Demand (MWh)", legend=:topright, 
    title="commercial Demand", ylim=(y_min, y_max), yformatter = :plain)
plot!(results_df[!, :time], results_df[!, :commercial_demand], label="Model-Solved commercial demand (MWh)")

UndefVarError: UndefVarError: `toy_data` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [27]:
# Plot actual vs model-solved industrial demand
plot(1:length(toy_data.industrial_demand_mwh), toy_data.industrial_demand_mwh, 
    label="Actual industrial demand (MWh)", xlabel="Time (hours)", ylabel="Demand (MWh)", legend=:topright, 
    title="Industrial Demand", ylim=(y_min, y_max), yformatter = :plain)
plot!(results_df[!, :time], results_df[!, :industrial_demand], label="Model-Solved industrial demand (MWh)")

UndefVarError: UndefVarError: `toy_data` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [28]:
# Plot actual coal vs model-solved coal across time
plot(1:length(toy_data.coal_gen_mwh), toy_data.coal_gen_mwh, 
label="Actual generation (MWh)", xlabel="Time (hours)", ylabel="Coal generation (MWh)", legend=:topright)
plot!(results_df[!, :time], results_df[!, :coal_gen], label="Model-Solved generation (MWh)")

UndefVarError: UndefVarError: `toy_data` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [29]:
# Plot actual coal vs model-solved coal across time
plot(1:length(toy_data.cogeneration_gen_mwh), toy_data.cogeneration_gen_mwh, 
label="Actual generation (MWh)", xlabel="Time (hours)", ylabel="Cogeneration generation (MWh)", legend=:topright)
plot!(results_df[!, :time], results_df[!, :cogeneration_gen], label="Model-Solved generation (MWh)")

UndefVarError: UndefVarError: `toy_data` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [30]:
# Plot actual combined cycle generation vs model-solved combined cycle generation across time
plot(1:length(toy_data.combined_cycle_gen_mwh), toy_data.combined_cycle_gen_mwh, 
  label="Actual generation (MWh)", xlabel="Time (hours)", ylabel="Natural gas generation (MWh)", yformatter = :plain, legend=:topright)
plot!(results_df[!, :time], results_df[!, :combined_cycle_gen], label="Model-Solved generation (MWh)")

UndefVarError: UndefVarError: `toy_data` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [31]:
# Plot actual combined cycle generation vs model-solved combined cycle generation across time
plot(1:length(toy_data.nonrenewable_waste_gen_mwh), toy_data.nonrenewable_waste_gen_mwh, 
  label="Actual generation (MWh)", xlabel="Time (hours)", ylabel="Non-renewable waste generation (MWh)", yformatter = :plain, legend=:topright)
plot!(results_df[!, :time], results_df[!, :non_renewable_waste_gen], label="Model-Solved generation (MWh)")

UndefVarError: UndefVarError: `toy_data` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [32]:
# Plot actual nuclear vs model-solved nuclear across time
plot(1:length(toy_data.nuclear_gen_mwh), toy_data.nuclear_gen_mwh, 
  label="Actual generation (MWh)", xlabel="Time (hours)", ylabel="Nuclear generation (MWh)", yformatter = :plain, legend=:topright)
plot!(results_df[!, :time], results_df[!, :nuclear_gen], label="Model-Solved generation (MWh)")

UndefVarError: UndefVarError: `toy_data` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [33]:
# Plot actual hydro vs model-solved conventional hydro across time
plot(1:length(toy_data.conventional_hydro_gen_mwh), toy_data.conventional_hydro_gen_mwh,
  label="Actual generation (MWh)", xlabel="Time (hours)", ylabel="Conventional hydro generation (MWh)", yformatter = :plain, legend=:topright)
plot!(results_df[!, :time], results_df[!, :conventional_hydro_gen], label="Model-Solved generation (MWh)")

UndefVarError: UndefVarError: `toy_data` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [34]:
# Plot actual run of river hydro vs model-solved run of river hydro across time
plot(1:length(toy_data.run_of_river_hydro_gen_mwh), toy_data.run_of_river_hydro_gen_mwh,
  label="Actual generation (MWh)", xlabel="Time (hours)", ylabel="Run of river hydro generation (MWh)", yformatter = :plain, legend=:topright)
plot!(results_df[!, :time], results_df[!, :run_of_river_hydro_gen], label="Model-Solved generation (MWh)")


UndefVarError: UndefVarError: `toy_data` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [35]:
# Plot actual pumped hydro vs model-solved pumped hydro across time with transparency for model line
plot(1:length(toy_data.pumped_hydro_gen_mwh), toy_data.pumped_hydro_gen_mwh,
    label="Actual generation (MWh)", xlabel="Time (hours)", ylabel="Pumped hydro generation (MWh)", yformatter = :plain, legend=:topright)
plot!(results_df[!, :time], results_df[!, :pumped_hydro_gen], label="Model-Solved generation (MWh)", alpha=0.5)

UndefVarError: UndefVarError: `toy_data` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [36]:
# Plot actual solar_pv vs model-solved solar_pv across time
plot(1:length(toy_data.solar_pv_gen_mwh), toy_data.solar_pv_gen_mwh, 
  label="Actual generation (MWh)", xlabel="Time (hours)", ylabel="Solar PV generation (MWh)", yformatter = :plain, legend=:topright)
plot!(results_df[!, :time], results_df[!, :solar_pv_gen], label="Model-Solved generation (MWh)")

UndefVarError: UndefVarError: `toy_data` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [37]:
# Plot actual solar_thermal vs model-solved solar_thermal across time
plot(1:length(toy_data.solar_thermal_gen_mwh), toy_data.solar_thermal_gen_mwh, title="Solar thermal generation",
  label="Actual generation (MWh)", xlabel="Time (hours)", ylabel="Solar thermal generation (MWh)", yformatter = :plain, legend=:topright)
plot!(results_df[!, :time], results_df[!, :solar_thermal_gen], label="Model-Solved generation (MWh)")

UndefVarError: UndefVarError: `toy_data` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [38]:
# Plot actual wind vs model-solved wind across time
plot(1:length(toy_data.wind_gen_mwh), toy_data.wind_gen_mwh, title="Wind generation",
  label="Actual generation (MWh)", xlabel="Time (hours)", ylabel="Wind generation (MWh)", yformatter = :plain, legend=:topright)
plot!(results_df[!, :time], results_df[!, :wind_gen], label="Model-Solved generation (MWh)")

UndefVarError: UndefVarError: `toy_data` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [39]:
# Plot actual other renewable vs model-solved other renewable across time
plot(1:length(toy_data.other_renewable_gen_mwh), toy_data.other_renewable_gen_mwh, label="Actual generation (MWh)", xlabel="Time (hours)", ylabel="Other renewable generation (MWh)", yformatter = :plain, legend=:topright)
plot!(results_df[!, :time], results_df[!, :other_renewable_gen], label="Model-Solved generation (MWh)")

UndefVarError: UndefVarError: `toy_data` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [40]:
# Plot actual renewable waste vs model-solved renewable waste across time
plot(1:length(toy_data.renewable_waste_gen_mwh), toy_data.renewable_waste_gen_mwh, label="Actual generation (MWh)", xlabel="Time (hours)", ylabel="Renewable waste generation (MWh)", yformatter = :plain, legend=:topright)
plot!(results_df[!, :time], results_df[!, :renewable_waste_gen], label="Model-Solved generation (MWh)")

UndefVarError: UndefVarError: `toy_data` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [41]:
# Plot actual battery use vs model-solved battery use across time
plot(1:length(toy_data.batteries_gen_mwh), toy_data.batteries_gen_mwh, label="Actual generation (MWh)", xlabel="Time (hours)", ylabel="Batteries generation (MWh)", yformatter = :plain, legend=:topright)
plot!(results_df[!, :time], results_df[!, :battery_gen], label="Model-Solved generation (MWh)")

UndefVarError: UndefVarError: `toy_data` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [42]:
# Plot actual imports from France vs model-solved imports from France across time
plot(1:length(toy_data.imports_France_mwh), toy_data.imports_France_mwh, 
  label="Actual imports (MWh)", xlabel="Time (hours)", ylabel="Imports from France (MWh)", yformatter = :plain, legend=:topright)
plot!(1:length(results_df[!, :time]), results_df[!, :imports_FRA], label="Model-Solved imports (MWh)")

UndefVarError: UndefVarError: `toy_data` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [43]:
# Plot actual imports from Portugal vs model-solved imports from Portugal across time
plot(1:length(toy_data.imports_Portugal_mwh), toy_data.imports_Portugal_mwh, 
  label="Actual imports (MWh)", xlabel="Time (hours)", ylabel="Imports from Portugal (MWh)", yformatter = :plain, legend=:topright)
plot!(1:length(results_df[!, :time]), results_df[!, :imports_POR], label="Model-Solved imports (MWh)")

UndefVarError: UndefVarError: `toy_data` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [44]:
# Plot actual imports from Morocco vs model-solved imports from Morocco across time
plot(1:length(toy_data.imports_Morocco_mwh), toy_data.imports_Morocco_mwh, 
  label="Actual imports (MWh)", xlabel="Time (hours)", ylabel="Imports from Morocco (MWh)", yformatter = :plain, legend=:topright)
plot!(1:length(results_df[!, :time]), results_df[!, :imports_MOR], label="Model-Solved imports (MWh)")

UndefVarError: UndefVarError: `toy_data` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [45]:
# Plot actual exports to France vs model-solved exports to France across time
plot(1:length(toy_data.exports_France_mwh), toy_data.exports_France_mwh, 
  label="Actual exports (MWh)", xlabel="Time (hours)", ylabel="Exports to France (MWh)", yformatter = :plain, legend=:topright)
plot!(1:length(results_df[!, :time]), results_df[!, :exports_FRA], label="Model-Solved exports (MWh)")

UndefVarError: UndefVarError: `toy_data` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [46]:
# Plot actual exports from Portugal vs model-solved exports from Portugal across time
plot(1:length(toy_data.exports_Portugal_mwh), toy_data.exports_Portugal_mwh, 
  label="Actual exports (MWh)", xlabel="Time (hours)", ylabel="Exports to Portugal (MWh)", yformatter = :plain, legend=:topright)
plot!(1:length(results_df[!, :time]), results_df[!, :exports_POR], label="Model-Solved exports (MWh)")

UndefVarError: UndefVarError: `toy_data` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [47]:
# Plot actual exports from Morocco vs model-solved exports from Morocco across time
plot(1:length(toy_data.exports_Morocco_mwh), toy_data.exports_Morocco_mwh, 
  label="Actual exports (MWh)", xlabel="Time (hours)", ylabel="Exports to Morocco (MWh)", yformatter = :plain, legend=:topright)
plot!(1:length(results_df[!, :time]), results_df[!, :exports_MOR], label="Model-Solved exports (MWh)")

UndefVarError: UndefVarError: `toy_data` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [48]:
# Plot actual share of renewable generation vs model-solved share of renewable generation across time
plot(1:length(toy_data.share_renewable_gen), toy_data.share_renewable_gen, 
  label="Actual share of renewable generation", xlabel="Time (hours)", ylabel="Share of renewable generation", yformatter = :plain, legend=:topright)
plot!(1:length(results_df[!, :time]), results_df[!, :share_renewable_gen], label="Model-Solved share of renewable generation")   

# Add a line which is the mean share of renewable generation in the plot
mean_share = mean(results_df[!, :share_renewable_gen])
plot!(1:length(results_df[!, :time]), fill(mean_share, length(results_df[!, :time])), label="Mean Share", linestyle=:dash, color=:red)

UndefVarError: UndefVarError: `toy_data` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [49]:
mean(toy_data.cost_coal_eur_mwh), mean(toy_data.cost_gas_eur_mwh), mean(toy_data.cost_diesel_eur_mwh), mean(toy_data.cost_uranium_eur_mwh)

UndefVarError: UndefVarError: `toy_data` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [50]:
# Marginal cost of coal
63 * 0.82 + 8.55 + 16.34 / 0.36

105.5988888888889

In [51]:
# Marginal cost of gas
63 * 0.49 + 2.29 + 46.54 / 0.57

114.80912280701756

In [52]:
mean(toy_data.coal_gen_mwh), minimum(toy_data.coal_gen_mwh)

UndefVarError: UndefVarError: `toy_data` not defined in `Main`
Suggestion: check for spelling errors or missing imports.